# 01 — Dataset download, validation, and few-shot preparation

**Purpose:** Download the official SIB-200 English/Urdu splits, preserve source IDs, detect leakage, and create repeatable nested training subsets.

**Platform:** Kaggle or Colab (Python + GPU for model stages). Never use held-out test data for model selection.

**Start here:** The first code cell clones the public GitHub repository automatically when needed. Enable Internet in Kaggle/Colab.

In [ ]:
# GitHub setup: run this as the FIRST code cell in Kaggle or Colab.
# This clones the source code only; GPU checkpoints/results need separate storage.
from pathlib import Path
import os, sys, subprocess

REPO_URL = "https://github.com/muhammad-omer-dev/nlp_research_proj.git"
REPO_FOLDER = "nlp_research_proj"
PROJECT_FOLDER = "Urdu_Qwen_Research_Notebooks"
PULL_LATEST = False  # True: update an existing clean clone from GitHub

if Path("/kaggle/working").is_dir():
    workspace = Path("/kaggle/working")
elif Path("/content").is_dir():
    workspace = Path("/content")
else:
    workspace = Path.cwd()

repo_dir = workspace / REPO_FOLDER
project_dir = repo_dir / PROJECT_FOLDER

# Local development support: use the existing source folder outside Kaggle/Colab.
local_project = next((p for p in [Path.cwd(), *Path.cwd().parents]
                      if (p / "project_utils.py").is_file()), None)
if local_project is not None and not (Path("/kaggle/working").is_dir() or Path("/content").is_dir()):
    ROOT = local_project
else:
    if not repo_dir.exists():
        print("Cloning:", REPO_URL)
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(repo_dir)])
    elif PULL_LATEST:
        # Only fast-forward; protects existing local changes from being overwritten.
        subprocess.check_call(["git", "-C", str(repo_dir), "pull", "--ff-only"])
    else:
        print("Using existing clone:", repo_dir)

    ROOT = project_dir

if not (ROOT / "project_utils.py").is_file():
    raise FileNotFoundError(
        f"Cannot find project_utils.py at {ROOT}. Confirm the GitHub folder name."
    )

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from project_utils import *
folders()
print("Project loaded from:", ROOT)


**Rules:** Do not alter or resplit the official test data; do not use the paper's 1,000 test headlines for training. Keep Urdu and English labels identical, save source IDs, and verify that no corresponding sentence appears in different splits. Datasets are downloaded from [Davlan/SIB-200](https://huggingface.co/datasets/Davlan/sib200).

In [ ]:
from datasets import load_dataset
from collections import Counter
TASK = 'sib200'
LANGS = {'ur': 'urd_Arab','en':'eng_Latn'}
DATA = {}
for lang, lang_id in LANGS.items():
    DATA[lang] = load_dataset('Davlan/sib200', lang_id)
    print(lang, {s: len(DATA[lang][s]) for s in DATA[lang]})
    print('fields:',DATA[lang]['train'].column_names)
    print('label feature:',DATA[lang]['train'].features['label'])

In [ ]:
labels = list(DATA['ur']['train'].features['label'].names)
assert labels == DEFAULT_LABELS, f'Unexpected label ordering: {labels}; inspect before proceeding.'
assert list(DATA['en']['train'].features['label'].names) == labels, 'English/Urdu labels differ'
rows = []
for lang, dsd in DATA.items():
    for split in ['train','validation','test']:
        for i, r in enumerate(dsd[split]):
            item = dict(example_id=stable_id(TASK, lang, split, str(r['index_id'])),
                        index_id=str(r['index_id']), source='Davlan/sib200', lang=lang, split=split,
                        text=str(r['text']), label_id=int(r['label']), label=labels[int(r['label'])])
            rows.append(item)
problems=validate_split(rows,labels)
print('Rows:',len(rows), 'issues:',problems[:10])
assert not problems, 'Inspect split leakage, missing values or duplicate IDs before proceeding.'
for s in ['train','validation','test']:
    en = {r['index_id']: r['label_id'] for r in rows if r['lang']=='en' and r['split']==s}
    ur = {r['index_id']: r['label_id'] for r in rows if r['lang']=='ur' and r['split']==s}
    assert en==ur, 'Parallel English and Urdu IDs/labels do not match; inspect before continuing.'
print('Parallel languages and labels match by index_id across all splits.')

In [ ]:
out = ROOT/'data'/TASK; out.mkdir(exist_ok=True)
for lang in LANGS:
    for split in ['train','validation','test']:
        group = [r for r in rows if r['lang']==lang and r['split']==split]
        write_csv(out / f'{lang}_{split}.csv', group)
counts = {lang: {s:Counter(r['label'] for r in rows if r['lang']==lang and r['split']==s)
                 for s in ['train','validation','test']} for lang in LANGS}
manifest = {'task':TASK, 'dataset':'Davlan/sib200','language_configs':LANGS, 'labels':labels,
            'rows_by_split': {f'{lang}_{split}':len([r for r in rows if r['lang']==lang and r['split']==split])
                              for lang in LANGS for split in ['train','validation','test']},
            'sha256':{f'{lang}_{split}':file_hash(out/f'{lang}_{split}.csv')
                      for lang in LANGS for split in ['train','validation','test']},
            'notes':'Fixed official splits; class mapping taken from downloaded ClassLabel features. Do not change test labels.'}
write_json(out/'manifest.json',manifest)
print('Saved:',out)
for lang in LANGS:
    print(lang,'class counts:',{split:dict(c) for split,c in counts[lang].items()})

In [ ]:
for seed in [42,123,456]:
    subsets = stratified_prefix_samples(load_task(TASK,'ur','train'), [5,10,25,50,100], seed=seed)
    for n, entry in subsets.items():
        if entry['available']:
            write_csv(out / f'ur_train_{n}perclass_seed{seed}.csv', entry['rows'])
            print(f'seed={seed} n/class={n} total={len(entry["rows"])}')
        else:
            print(f'seed={seed} n/class={n} NOT available:', entry['shortfall'])
print('Important: 100 examples per class is not guaranteed; do not oversample without explicitly reporting it.')